# 07 — Reproduksi non-neural end-to-end (notebook mandiri)

Seluruh kode feature engineering, `Pipeline`, `ColumnTransformer`, training, audit, full refit, dan inferensi dari model tersimpan dicantumkan di notebook ini. **Input cukup ZIP kompetisi** dan dependency Python. Model ini merupakan **eksperimen baru**, bukan klaim rekonstruksi tepat submission 010/011 yang checkpoint-nya tidak lengkap.


In [ ]:
"""Target-independent building, material and geographic feature construction."""
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin


class SeismicFeatures(BaseEstimator, TransformerMixin):
    """Create seismic-vulnerability proxies without using labels or test statistics.

    Full building IDs are never passed to a classifier. The shorter prefixes
    are optional spatial proxies; evaluate with and without them to detect
    reliance on memorized geographic patterns.
    """

    def __init__(self, use_geo=True):
        """Set whether coarse ID prefixes and reference-frequency features are enabled."""
        self.use_geo = use_geo

    def fit(self, X, y=None):
        """Learn unsupervised prefix counts using only the training partition."""
        ids = X["building_id"].astype("string").fillna("").str.zfill(12)
        self.freq_ = {}
        if self.use_geo:
            for n in (4, 6):
                self.freq_[n] = ids.str[:n].value_counts().to_dict()
        return self

    def transform(self, X):
        """Return a fresh DataFrame with safe ratios, counts and material flags."""
        x = X.copy().reset_index(drop=True)
        x = x.drop(columns=["damage_grade"], errors="ignore")
        geography = [c for c in ("district_id", "vdcmun_id", "ward_id") if c in x]
        if not self.use_geo:
            x = x.drop(columns=geography)
        else:
            for c in geography:
                x[c] = x[c].astype(str)
        ids = x.pop("building_id").astype("string").fillna("").str.zfill(12)
        for name in list(x.columns):
            if name.startswith("has_"):
                x[name] = pd.to_numeric(x[name], errors="coerce")
        numeric = ("age_building", "plinth_area_sq_ft", "height_ft_pre_eq",
                   "count_floors_pre_eq", "count_floors_post_eq")
        for name in numeric:
            if name in x:
                x[name] = pd.to_numeric(x[name], errors="coerce")
        if "count_floors_post_eq" in x:
            # Post-earthquake attributes could leak damage severity.
            x = x.drop(columns=["count_floors_post_eq"])
        if "height_ft_post_eq" in x:
            x = x.drop(columns=["height_ft_post_eq"])
        pre = x.get("count_floors_pre_eq", pd.Series(np.nan, index=x.index))
        height = x.get("height_ft_pre_eq", pd.Series(np.nan, index=x.index))
        area = x.get("plinth_area_sq_ft", pd.Series(np.nan, index=x.index))
        age = x.get("age_building", pd.Series(np.nan, index=x.index))
        x["pre_height_per_floor"] = height / pre.replace(0, np.nan)
        x["plinth_per_floor"] = area / pre.replace(0, np.nan)
        x["age_log"] = np.log1p(age.clip(lower=0))
        x["plinth_log"] = np.log1p(area.clip(lower=0))
        x["age_x_floors"] = age * pre
        flag_cols = [c for c in x if c.startswith("has_superstructure_")]
        if flag_cols:
            x["n_superstructure_types"] = x[flag_cols].fillna(0).sum(axis=1)
        if self.use_geo:
            for n in (2, 4, 6):
                x[f"id_prefix_{n}"] = ids.str[:n].astype(str)
            for n in (4, 6):
                x[f"id_prefix_{n}_reference_count"] = (
                    ids.str[:n].map(self.freq_[n]).fillna(0).astype(float)
                )
        return x.replace([np.inf, -np.inf], np.nan)


# NEXT MODULE

"""Input contracts and scikit-learn ColumnTransformer pipelines."""
from pathlib import Path
from zipfile import ZipFile
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder


def read_competition_data(path="data"):
    """Load train/test/sample from a folder or ZIP (including nested ZIP paths)."""
    path = Path(path)
    required = ("train.csv", "test.csv", "sample_submission.csv")
    if path.is_file() and path.suffix.lower() == ".zip":
        with ZipFile(path) as archive:
            mapping = {}
            for item in archive.namelist():
                name = Path(item).name.lower()
                if name in required:
                    if name in mapping:
                        raise ValueError(f"Duplicate {name} inside ZIP")
                    mapping[name] = item
            missing = set(required) - set(mapping)
            if missing:
                raise FileNotFoundError(f"Missing from ZIP: {sorted(missing)}")
            tables = {n: pd.read_csv(archive.open(mapping[n]), dtype={"building_id": str})
                      for n in required}
    else:
        if path.is_dir():
            names = {p.name.lower(): p for p in path.rglob("*.csv")}
            missing = set(required) - set(names)
            if missing:
                raise FileNotFoundError(f"Missing from {path}: {sorted(missing)}. Download data from Kaggle first.")
            tables = {n: pd.read_csv(names[n], dtype={"building_id": str}) for n in required}
        else:
            raise FileNotFoundError(f"Data source does not exist: {path}")
    train, test, sample = (tables[n] for n in required)
    for title, frame in (("train", train), ("test", test), ("sample", sample)):
        if "building_id" not in frame or frame.building_id.isna().any() or not frame.building_id.is_unique:
            raise ValueError(f"{title}: building_id missing, null, or duplicate")
    if "damage_grade" not in train or "damage_grade" not in sample:
        raise ValueError("Missing target 'damage_grade'")
    if "damage_grade" in test:
        raise ValueError("Test must not contain target")
    if set(train.columns) - {"damage_grade"} != set(test.columns):
        raise ValueError("Train and test feature columns differ")
    if len(test) != len(sample) or set(test.building_id) != set(sample.building_id):
        raise ValueError("Sample IDs must exactly match test IDs")
    if set(train.building_id) & set(test.building_id):
        raise ValueError("Train/test building_id overlap")
    return train, test, sample


def parse_grades(series):
    """Accept integer classes 1..5 or strings 'Grade 1'..'Grade 5'."""
    parsed = series.astype("string").str.strip().str.extract(
        r"^(?:Grade\s*)?([1-5])$", expand=False
    )
    if parsed.isna().any():
        raise ValueError(f"Unexpected damage grade values: {series[parsed.isna()].head().tolist()}")
    return parsed.astype(int).to_numpy()


class ToText(BaseEstimator, TransformerMixin):
    """Convert missing categorical features to stable strings for CatBoost."""

    def fit(self, X, y=None):
        """No learned parameters are required."""
        return self

    def transform(self, X):
        """Convert categorical input to 2D numpy string array."""
        return pd.DataFrame(X).fillna("__missing__").astype(str).to_numpy(dtype=object)


def make_preprocessor(X_reference, family="lgbm"):
    """Construct a fitted-at-training-time ColumnTransformer and category positions."""
    nums = [c for c in X_reference if pd.api.types.is_numeric_dtype(X_reference[c])]
    cats = [c for c in X_reference if c not in nums]
    numeric = Pipeline([("impute", SimpleImputer(strategy="median", add_indicator=False))])
    if family == "catboost":
        categorical = Pipeline([("text", ToText())])
    else:
        categorical = Pipeline([
            ("impute", SimpleImputer(strategy="constant", fill_value="__missing__")),
            ("encode", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
        ])
    transformers = []
    if nums:
        transformers.append(("numeric", numeric, nums))
    if cats:
        transformers.append(("categorical", categorical, cats))
    return ColumnTransformer(transformers, remainder="drop", sparse_threshold=0), list(range(len(nums), len(nums) + len(cats)))


def build_feature_pipeline(X_reference, estimator, family="lgbm", use_geo=True):
    """Combine feature generation, ColumnTransformer and estimator in one Pipeline."""
    feats = SeismicFeatures(use_geo=use_geo).fit_transform(X_reference)
    columns, cat_indices = make_preprocessor(feats, family)
    if family == "catboost":
        estimator.set_params(cat_features=cat_indices)
    return Pipeline([("features", SeismicFeatures(use_geo=use_geo)),
                     ("columns", columns), ("model", estimator)])


def build_submission(sample, test, predicted, output, label_style="numeric"):
    """Align predictions by building_id; assert order, values and schema before saving."""
    if len(test) != len(predicted):
        raise ValueError("Prediction length differs from test")
    values = np.asarray(predicted)
    if values.ndim != 1 or not np.isin(values, np.arange(1, 6)).all():
        raise ValueError("Predictions must be integer-valued grades 1..5")
    if (sample.building_id.isna().any() or not sample.building_id.is_unique
            or len(sample) != len(test)
            or set(sample.building_id.astype(str)) != set(test.building_id.astype(str))):
        raise ValueError("Sample IDs must uniquely and exactly match test IDs")
    lookup = pd.Series(values.astype(int), index=test.building_id.astype(str))
    if not lookup.index.is_unique:
        raise ValueError("Duplicate test IDs")
    result = pd.DataFrame({"building_id": sample.building_id.astype(str)})
    result["damage_grade"] = result.building_id.map(lookup)
    if result.damage_grade.isna().any() or not result.damage_grade.isin(range(1, 6)).all():
        raise ValueError("Submission ID mismatch or invalid grade")
    result["damage_grade"] = result.damage_grade.astype(int)
    if label_style == "auto":
        examples = sample["damage_grade"].dropna().astype(str).str.strip()
        label_style = "grade" if len(examples) and examples.str.fullmatch(r"Grade [1-5]").all() else "numeric"
    if label_style == "grade":
        result["damage_grade"] = "Grade " + result.damage_grade.astype(str)
    elif label_style != "numeric":
        raise ValueError("label_style must be 'auto', 'numeric' or 'grade'")
    output = Path(output)
    output.parent.mkdir(parents=True, exist_ok=True)
    result.to_csv(output, index=False)
    reread = pd.read_csv(output, dtype={"building_id": str})
    pd.testing.assert_frame_equal(reread, result)
    return result


# NEXT MODULE

"""Reproducible Macro-F1 scoring and calibration using only calibration labels."""
import json
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, f1_score


CLASSES = np.array([1, 2, 3, 4, 5])


def macro_f1(y, predictions):
    """Return unweighted average F1 across exactly five classes."""
    return float(f1_score(y, predictions, labels=CLASSES, average="macro", zero_division=0))


def decide(probabilities, offsets=None):
    """Apply per-class log-probability offsets and return class labels."""
    p = np.asarray(probabilities)
    if p.ndim != 2 or p.shape[1] != 5:
        raise ValueError("Expected probability matrix with five columns")
    offsets = np.zeros(5) if offsets is None else np.asarray(offsets, dtype=float)
    return CLASSES[np.argmax(np.log(np.clip(p, 1e-12, 1)) + offsets[None, :], axis=1)]


def tune_offsets(probabilities, y, seed=42, passes=4):
    """Coordinate-search offsets on calibration labels only; keep zero-offset control."""
    rng = np.random.default_rng(seed)
    candidates = [np.zeros(5)]
    for _ in range(3):
        candidates.append(rng.normal(0, 0.15, size=5))
    best = np.zeros(5)
    best_score = macro_f1(y, decide(probabilities, best))
    for starting in candidates:
        trial = starting.copy()
        for width in (0.35, 0.16, 0.07, 0.03)[:passes]:
            changed = True
            steps = 0
            while changed and steps < 3:
                changed = False
                steps += 1
                for cls in rng.permutation(5):
                    for delta in (-width, width):
                        temp = trial.copy()
                        temp[cls] += delta
                        temp -= temp.mean()
                        score = macro_f1(y, decide(probabilities, temp))
                        # Require meaningful improvements to avoid noise fitting.
                        if score > best_score + 0.0001:
                            best_score, best, trial = score, temp.copy(), temp.copy()
                            changed = True
    return best, best_score


def experiment_report(y, pred, output, metadata=None):
    """Write class F1, classification report, confusion matrix and experiment context."""
    output = Path(output)
    output.parent.mkdir(parents=True, exist_ok=True)
    result = {
        "macro_f1": macro_f1(y, pred),
        "classification_report": classification_report(y, pred, labels=CLASSES,
                                                         output_dict=True, zero_division=0),
        "confusion_matrix": confusion_matrix(y, pred, labels=CLASSES).tolist(),
        "metadata": metadata or {}
    }
    output.write_text(json.dumps(result, indent=2), encoding="utf-8")
    return result


# NEXT MODULE

"""Train/tune/audit non-neural classifiers and export a competition-ready submission.

Run from repository root: python -m scripts.train --data data --quick
"""
import argparse
import json
import time
from pathlib import Path
import joblib
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier


def specs(quick=False, seed=42):
    """Define bounded LightGBM/CatBoost candidates; no neural networks."""
    rounds = 140 if quick else 950
    catrounds = 150 if quick else 1050
    common = dict(objective="multiclass", num_class=5, verbosity=-1, n_jobs=-1,
                  random_state=seed, max_depth=-1, subsample_freq=1)
    return [
        ("lgb_raw", "lgbm", True, LGBMClassifier(**common, n_estimators=rounds,
                learning_rate=0.045, num_leaves=47, min_child_samples=55,
                colsample_bytree=0.86, subsample=0.88, reg_lambda=3.0)),
        ("lgb_balanced", "lgbm", True, LGBMClassifier(**common, n_estimators=rounds,
                learning_rate=0.035, num_leaves=31, min_child_samples=80,
                colsample_bytree=0.9, reg_lambda=5.0, class_weight="balanced")),
        ("lgb_no_geo", "lgbm", False, LGBMClassifier(**common, n_estimators=rounds,
                learning_rate=0.04, num_leaves=47, min_child_samples=60,
                colsample_bytree=0.88, reg_lambda=3.0)),
        ("cat_native", "catboost", True, CatBoostClassifier(iterations=catrounds,
                depth=7, learning_rate=0.05, loss_function="MultiClass", l2_leaf_reg=6,
                random_seed=seed, verbose=False, thread_count=-1,
                allow_writing_files=False))
    ]


def ordered_probabilities(model, X):
    """Map estimator class order into columns for Grades 1..5."""
    predictions = np.asarray(model.predict_proba(X), dtype=float)
    classes = np.asarray(model.named_steps["model"].classes_, dtype=int)
    result = np.zeros((len(X), 5), dtype=float)
    for i, grade in enumerate(classes):
        result[:, int(grade) - 1] = predictions[:, i]
    return result


def candidates(probs, y_cal):
    """Evaluate individual models and simple blends without touching audit labels."""
    names = list(probs)
    out = []
    for name in names:
        out.append((name, {name: 1.0}, probs[name]))
    ranked = sorted(names, key=lambda name: macro_f1(y_cal, decide(probs[name])), reverse=True)
    for a in ranked[:3]:
        for b in ranked[:3]:
            if a >= b:
                continue
            for weight in (0.35, 0.5, 0.65):
                out.append((f"{a}_{b}_{weight}", {a: weight, b: 1-weight},
                            weight * probs[a] + (1-weight) * probs[b]))
    return out


def train(args):
    """Keep calibration/model selection separate from a single untouched audit holdout."""
    train_df, test_df, sample = read_competition_data(args.data)
    y = parse_grades(train_df["damage_grade"])
    X = train_df.drop(columns=["damage_grade"])
    if not np.array_equal(np.unique(y), CLASSES):
        raise ValueError("Expected all five training labels 1..5")
    idx = np.arange(len(y))
    dev, audit = train_test_split(idx, test_size=0.15, stratify=y, random_state=args.seed)
    fit_idx, cal = train_test_split(dev, test_size=0.15 / 0.85, stratify=y[dev],
                                    random_state=args.seed + 1)
    out = Path(args.output)
    out.mkdir(parents=True, exist_ok=True)
    print(f"Train={len(fit_idx)} calibration={len(cal)} untouched_audit={len(audit)}")
    predictions_cal, predictions_audit = {}, {}
    definitions = {name:(family, geo, estimator)
                   for name, family, geo, estimator in specs(args.quick, args.seed)}
    t0 = time.monotonic()
    for name, (family, geo, estimator) in definitions.items():
        pipe = build_feature_pipeline(X.iloc[fit_idx], estimator, family, geo)
        pipe.fit(X.iloc[fit_idx], y[fit_idx])
        predictions_cal[name] = ordered_probabilities(pipe, X.iloc[cal])
        predictions_audit[name] = ordered_probabilities(pipe, X.iloc[audit])
        print(f"{name}: CAL Macro-F1={macro_f1(y[cal], decide(predictions_cal[name])):.6f}")
    leaderboard = []
    for name, weights, p in candidates(predictions_cal, y[cal]):
        raw = macro_f1(y[cal], decide(p))
        offsets, adjusted = tune_offsets(p, y[cal], seed=args.seed, passes=2 if args.quick else 4)
        # Prefer uncalibrated decision if offset gain is too small.
        if adjusted < raw + 0.0015:
            offsets, adjusted = np.zeros(5), raw
        leaderboard.append((adjusted, name, weights, offsets, raw))
    leaderboard.sort(key=lambda item: item[0], reverse=True)
    score_cal, winner, weights, offsets, raw = leaderboard[0]
    # Audit labels are referenced ONLY AFTER the choice is frozen.
    p_audit = sum(weight * predictions_audit[name] for name, weight in weights.items())
    p_baseline = predictions_audit["lgb_raw"]
    audit_baseline = macro_f1(y[audit], decide(p_baseline))
    audit_candidate = macro_f1(y[audit], decide(p_audit, offsets))
    audit_raw = macro_f1(y[audit], decide(p_audit))
    print(f"Selected={winner} CAL={score_cal:.6f} AUDIT={audit_candidate:.6f}"
          f" (raw_audit={audit_raw:.6f}, lgb_baseline_audit={audit_baseline:.6f})")
    experiment_report(y[audit], decide(p_audit, offsets), out/"audit_metrics.json", {
        "winner": winner, "calibration_f1":score_cal,
        "audit_lgb_raw_f1":audit_baseline, "audit_uncalibrated_f1":audit_raw,
        "selection_used_audit_labels":False, "quick":args.quick,
        "rows_fit":len(fit_idx), "rows_cal":len(cal), "rows_audit":len(audit),
        "train_seconds":round(time.monotonic()-t0, 1)
    })
    selection = {"name":winner, "weights":weights, "offsets":offsets.tolist(),
                 "cal_f1":score_cal, "audit_f1":audit_candidate,
                 "baseline_audit_f1":audit_baseline,
                 "all_cal_scores":[{"name":n, "cal_score":s, "raw_score":r}
                    for s,n,w,o,r in leaderboard]}
    (out/"selection.json").write_text(json.dumps(selection,indent=2), encoding="utf-8")
    # Refit using all available labeled rows, never using hidden Kaggle targets.
    models, p_test = {}, np.zeros((len(test_df), 5), dtype=float)
    for name, weight in weights.items():
        family, geo, estimator = definitions[name]
        pipe = build_feature_pipeline(X, estimator, family, geo)
        pipe.fit(X, y)
        p_test += weight * ordered_probabilities(pipe, test_df)
        models[name] = pipe
    predictions = decide(p_test, offsets)
    build_submission(sample, test_df, predictions, out/"submission.csv", "numeric")
    joblib.dump({"models":models, "weights":weights, "offsets":offsets.tolist(),
                 "classes":CLASSES.tolist()}, out/"model_bundle.joblib")
    print(f"Saved: {out/'submission.csv'} | Models: {out/'model_bundle.joblib'}")
    return selection


def main():
    """Parse CLI arguments and execute the reproducible experiment."""
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--data", default="data", help="Kaggle CSV folder or competition ZIP")
    parser.add_argument("--output", default="submissions")
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--quick", action="store_true", help="Short smoke run, not final scoring")
    train(parser.parse_args())




# NEXT MODULE

"""Load a trained ensemble and rebuild a format-validated Kaggle submission."""
import argparse
from pathlib import Path
import joblib
import numpy as np


def predict(data, model_path, output="submissions/submission.csv", label_style="numeric"):
    """Predict with saved fitted Pipelines; preserve sample_submission row order."""
    _, test, sample = read_competition_data(data)
    bundle = joblib.load(model_path)
    p = np.zeros((len(test), 5), dtype=float)
    for name, weight in bundle["weights"].items():
        p += weight * ordered_probabilities(bundle["models"][name], test)
    predictions = decide(p, bundle["offsets"])
    return build_submission(sample, test, predictions, Path(output), label_style)


def main():
    """Provide a command-line interface for model inference."""
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--data", default="data")
    parser.add_argument("--model", default="submissions/model_bundle.joblib")
    parser.add_argument("--output", default="submissions/submission.csv")
    parser.add_argument("--label-style", choices=["auto", "numeric", "grade"], default="numeric")
    args = parser.parse_args()
    result = predict(args.data, args.model, args.output, args.label_style)
    print(result.head(), f"\nSaved {len(result)} rows to {args.output}")




## Dataset dan EDA

Gunakan ZIP di folder data dari repository atau atur `DATA_SOURCE`. `QUICK=False` berarti full model; `True` hanya tes alur. Notebook tidak mengimpor file apa pun dari folder scripts.


In [ ]:
from pathlib import Path
import argparse, json, hashlib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
HERE=Path.cwd()
OPTIONS=[HERE/'data'/'structural-damage-classification.zip', HERE.parent/'data'/'structural-damage-classification.zip', HERE/'structural-damage-classification.zip']
if Path('/kaggle/input').exists():
    OPTIONS.extend(Path('/kaggle/input').rglob('*.zip'))
DATA_SOURCE=next((p for p in OPTIONS if p.is_file()), None)
if DATA_SOURCE is None: raise FileNotFoundError('Dataset ZIP tidak ditemukan: set DATA_SOURCE secara manual')
print('ZIP:',DATA_SOURCE)
data_train,data_test,sample=read_competition_data(DATA_SOURCE)
y_all=parse_grades(data_train.damage_grade)
print('Train/Test:',data_train.shape,data_test.shape,'Label distribution:',pd.Series(y_all).value_counts().sort_index().to_dict())
print('Missing train:',data_train.isna().sum().sum())
pd.Series(y_all).value_counts().sort_index().plot(kind='bar',title='Distribusi damage grade (train)')
plt.show()
QUICK=False
OUT=Path('artifacts/standalone_full')
OUT.mkdir(parents=True,exist_ok=True)


## Latih, pilih model pada CAL, uji pada AUDIT, lalu refit pada seluruh train

AUDIT tidak dipakai memilih konfigurasi pada run ini. Namun dataset sama sudah pernah digunakan adaptif secara historis sehingga ini bukan holdout independen yang murni. Tidak memakai neural network.


In [ ]:
selection=train(argparse.Namespace(data=str(DATA_SOURCE),output=str(OUT),seed=42,quick=QUICK))
print('Selected:',selection['name'],'CAL:',selection['cal_f1'],'AUDIT:',selection['audit_f1'])


## Confusion matrix lengkap: counts, normalisasi baris, precision, recall, F1 per kelas

Yang dianalisis adalah hasil **AUDIT lokal** atas label train yang diketahui, bukan hidden-label Kaggle test.


In [ ]:
from IPython.display import display
m=json.loads((OUT/'audit_metrics.json').read_text())
cm=np.asarray(m['confusion_matrix'])
assert cm.shape==(5,5)
ct=pd.DataFrame(cm,index=[f'Actual {k}' for k in range(1,6)],columns=[f'Pred {k}' for k in range(1,6)])
norm=ct.div(ct.sum(1),axis=0)
per_class=pd.DataFrame(m['classification_report']).T
display(ct);display(norm.round(3));display(per_class)
ct.to_csv(OUT/'confusion_counts.csv')
norm.to_csv(OUT/'confusion_row_normalized.csv')
per_class.to_csv(OUT/'per_class_report.csv')
for kind,frame in [('counts',ct),('row_normalized',norm)]:
    mat=frame.to_numpy()
    fig,ax=plt.subplots(figsize=(8,7));ax.imshow(mat,cmap='Blues')
    ax.set_xticks(range(5),range(1,6));ax.set_yticks(range(5),range(1,6))
    ax.set(xlabel='Predicted grade',ylabel='Actual grade',title=f'Audit confusion: {kind}')
    for i in range(5):
        for j in range(5):
            ax.text(j,i,f'{mat[i,j]:,.0f}' if kind=='counts' else f'{mat[i,j]:.1%}',ha='center',va='center')
    fig.tight_layout();fig.savefig(OUT/f'confusion_{kind}.png',dpi=160);plt.show()


## Replay dari checkpoint baru dan verifikasi CSV byte-identical

Hasil replay **harus sama persis** dengan file yang baru diproduksi. Ini mereproduksi inferensi kandidat baru, **bukan** file submission historis 010.


In [ ]:
original=OUT/'submission.csv'
replay=OUT/'submission_replay.csv'
submission=predict(str(DATA_SOURCE),str(OUT/'model_bundle.joblib'),str(replay),label_style='numeric')
assert original.read_bytes()==replay.read_bytes(),'CSV prediction replay mismatch'
assert submission.building_id.astype(str).equals(sample.building_id.astype(str))
assert submission.damage_grade.isin(range(1,6)).all()
manifest={'kind':'new_non_neural_candidate','byte_identical_replay':True,'data_zip':str(DATA_SOURCE),
'submission_sha256':hashlib.sha256(original.read_bytes()).hexdigest(),'rows':len(submission),
'audit_macro_f1':m['macro_f1'],'public_score':None,'private_score':None,'historic_010_reproduced':False}
(OUT/'repro_manifest.json').write_text(json.dumps(manifest,indent=2))
print(json.dumps(manifest,indent=2))
print('Jangan mengganti best 010 sebelum skor Kaggle kandidat baru terverifikasi.')
